# JEPA Setup Explainer (CIFAR-10)

**What this notebook shows.** A complete visual walk-through of what the JEPA training loop in
`JEPA_CIFAR10.ipynb` is actually optimizing. Each step is illustrated on a single CIFAR-10 image:

1. **Data.** How the fixed 40 % pool, the 20 % context chunk, and the 64 target coordinates are formed.
2. **Encoder.** What `latent_pos` is, how it grids the image, and how Gaussian attention bias makes each latent attend to its neighborhood.
3. **Target side.** The deterministic Gaussian soft-pool that builds the target features at the query coordinates.
4. **Predictor side.** The trained module that must guess those features from the 20 % chunk only.
5. **Loss.** Smooth-L1 between the predictor and the centered/LayerNormed target.

This is read-only: nothing trains. It just builds a fresh randomly-initialized model and visualizes
one forward pass so the components and their geometric relationships are explicit.


In [ ]:
import os, math, ssl, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

ssl._create_default_https_context = ssl._create_unverified_context
torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Config (matches JEPA_CIFAR10.ipynb) -----------------------------------
IMAGE_SIZE      = 32
N_PIX           = IMAGE_SIZE * IMAGE_SIZE          # 1024
FRAC_POOL       = 0.40
K_POOL          = int(round(FRAC_POOL * N_PIX))    # 410
K_HALF          = K_POOL // 2                      # 205
N_TGT           = 64
POS_BIAS_SIGMA  = 0.5
POS_POOL_SIGMA  = 0.15
FOURIER_DIM     = 96
POS_DIM         = FOURIER_DIM * 2                  # 192
INPUT_DIM       = 3 + POS_DIM                      # 195
LATENT_DIMS     = (256, 384, 512)
NUM_LATENTS     = (256, 256, 256)
EMBED_DIM       = LATENT_DIMS[-1]                  # 512
PRED_DIM        = 256
POOL_SEED       = 12345

print(f"K_POOL={K_POOL}  K_HALF={K_HALF}  N_TGT={N_TGT}  EMBED_DIM={EMBED_DIM}")


## Step 1 — The data

Each CIFAR-10 image has three structured views attached to it:

| View | Size | Who sees it |
|---|---|---|
| **40 % pool** | 410 pixels | the **target encoder** (fixed per image by seed) |
| **20 % context chunk** | 205 pixels | the **context encoder** (sampled fresh per iteration: nearest pool members to a random anchor) |
| **N_TGT target coords** | 64 positions | the **predictor**'s query points (sampled fresh per iteration) |

The 40 % is a *constraint*: the model is never allowed to see more. The 20 % chunk simulates the
test-time budget. The 64 target coordinates are where we ask "what's there?" and supervise the
predictor against the target encoder's view of those locations.


In [ ]:
# Load a single CIFAR-10 image (no augmentation, just normalize)
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,)*3, (0.5,)*3)])
cifar = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
classes = cifar.classes

# Pre-build the per-instance fixed 40% pool seeded as in training
rng = np.random.RandomState(POOL_SEED)
pool_idx_all = np.stack(
    [rng.permutation(N_PIX)[:K_POOL] for _ in range(len(cifar))],
    axis=0,
).astype(np.int64)

ys, xs = torch.meshgrid(
    torch.linspace(-1.0, 1.0, IMAGE_SIZE),
    torch.linspace(-1.0, 1.0, IMAGE_SIZE),
    indexing='ij',
)
coords_all = torch.stack([ys, xs], dim=-1).view(N_PIX, 2)


def build_sample(idx, anchor_seed=None, target_seed=None):
    """Return one (deterministic) sample with all the views attached."""
    img, label = cifar[idx]
    pix_all  = img.permute(1, 2, 0).reshape(-1, 3)           # (1024, 3) in [-1, 1]
    pool     = pool_idx_all[idx]                             # (410,) global indices
    pool_xy  = coords_all[pool]                              # (410, 2) coords in [-1, 1]²

    # Pick anchor and sample context: K_HALF nearest pool members
    anchor_rng = np.random.RandomState(anchor_seed if anchor_seed is not None else idx * 7 + 1)
    anchor_local = anchor_rng.randint(K_POOL)
    d2 = (pool_xy - pool_xy[anchor_local]).pow(2).sum(-1).numpy()
    ctx_local = np.argsort(d2, kind="stable")[:K_HALF]
    ctx_idx   = pool[ctx_local]

    # Sample N_TGT target coords (random subset of pool)
    tgt_rng = np.random.RandomState(target_seed if target_seed is not None else idx * 7 + 2)
    tgt_local = tgt_rng.choice(K_POOL, size=N_TGT, replace=False)
    tgt_idx   = pool[tgt_local]

    return {
        'image': img, 'label': int(label),
        'pix_all':  pix_all,
        'pool_idx': pool,             'pool_xy': coords_all[pool],
        'ctx_idx':  ctx_idx,          'ctx_xy':  coords_all[ctx_idx],
        'tgt_idx':  tgt_idx,          'tgt_xy':  coords_all[tgt_idx],
        'anchor_xy': pool_xy[anchor_local],
    }


# Pick an image
SAMPLE_IDX = 42
sample = build_sample(SAMPLE_IDX)
print(f"Sample: class='{classes[sample['label']]}' (idx={sample['label']})")


In [ ]:
# Visualize the four views: original / 40% pool / 20% context chunk / target coords

def coord_to_pixel(xy):
    """Map [-1, 1]² coord to image pixel index."""
    return ((xy + 1) / 2 * (IMAGE_SIZE - 1)).numpy()


img_show = (sample['image'].permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1).numpy()
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Panel 1: Original
axes[0].imshow(img_show, interpolation='nearest')
axes[0].set_title(f"Original\n'{classes[sample['label']]}' ({IMAGE_SIZE}×{IMAGE_SIZE} = {N_PIX} px)")
axes[0].axis('off')

# Panel 2: 40% pool — pool pixels in color, rest gray
pool_mask = np.zeros(N_PIX, dtype=bool); pool_mask[sample['pool_idx']] = True
panel2 = np.full((N_PIX, 3), 0.5)
panel2[pool_mask] = (sample['pix_all'][pool_mask].numpy() * 0.5 + 0.5).clip(0, 1)
axes[1].imshow(panel2.reshape(IMAGE_SIZE, IMAGE_SIZE, 3), interpolation='nearest')
axes[1].set_title(f"40% pool: {K_POOL} px\n(fixed per image, seeded)")
axes[1].axis('off')

# Panel 3: 20% context chunk (subset of pool nearest to a random anchor)
ctx_mask = np.zeros(N_PIX, dtype=bool); ctx_mask[sample['ctx_idx']] = True
panel3 = np.full((N_PIX, 3), 0.5)
panel3[ctx_mask] = (sample['pix_all'][ctx_mask].numpy() * 0.5 + 0.5).clip(0, 1)
axes[2].imshow(panel3.reshape(IMAGE_SIZE, IMAGE_SIZE, 3), interpolation='nearest')
anchor_pix = coord_to_pixel(sample['anchor_xy'])
axes[2].plot(anchor_pix[1], anchor_pix[0], '*', markersize=22, markerfacecolor='red',
             markeredgecolor='white', markeredgewidth=2)
axes[2].set_title(f"20% context chunk: {K_HALF} px\n(K_HALF nearest to anchor ★)")
axes[2].axis('off')

# Panel 4: target coords overlaid on full image
axes[3].imshow(img_show * 0.4 + 0.5, interpolation='nearest')
tgt_pix = coord_to_pixel(sample['tgt_xy'])
axes[3].scatter(tgt_pix[:, 1], tgt_pix[:, 0], s=40, c='red', marker='o',
                edgecolors='white', linewidths=1.2)
axes[3].set_title(f"{N_TGT} target coords\n(resampled per iteration)")
axes[3].axis('off')

plt.tight_layout(); plt.show()
print("\nNotice: the 40% pool is fixed across iterations; the 20% chunk and the 64 target coords")
print("are re-sampled every training step → predictor sees diverse (chunk, targets) pairs.")


## Step 2 — The encoder: content latents with learnable spatial anchors

OmniField's encoder is a Perceiver-style stack: 256 **latent tokens**, each carrying both
- a learnable **content** vector (initialized with sinusoidal embeddings)
- a learnable **2D anchor position** `latent_pos[l] ∈ ℝ²`, initialized on a 16 × 16 grid in [-1, 1]²

The two work together via a **Gaussian attention bias** added to each cross-attention block:

$$
\text{attn\_bias}[b, l, n] = -\frac{\| \text{latent\_pos}[l] - \text{input\_coord}[b, n] \|^2}{2\sigma^2}
$$

This biases latent `l` to attend preferentially to input pixels near its anchor — so `g[l]` ends up
representing "what's in the neighborhood of `latent_pos[l]`." With `σ = 0.5` (image coord units in
[-1, 1]²), each latent attends to roughly half the image's width around its anchor.


In [ ]:
# Build the latent_pos grid exactly as JEPAEncoder does at init
gs = max(2, int(math.ceil(math.sqrt(NUM_LATENTS[0]))))      # 16
ys = torch.linspace(-1.0, 1.0, gs)
xs = torch.linspace(-1.0, 1.0, gs)
grid_y = ys.unsqueeze(1).expand(gs, gs).contiguous()
grid_x = xs.unsqueeze(0).expand(gs, gs).contiguous()
latent_pos = torch.stack([grid_y, grid_x], dim=-1).reshape(-1, 2)
latent_pos = latent_pos + 0.01 * torch.randn_like(latent_pos)        # tiny jitter, same as training
print(f"latent_pos shape: {tuple(latent_pos.shape)}  var/axis = {latent_pos.var(dim=0).tolist()}")

# Visualize:
# Panel 1 — All 256 latent anchors overlaid on the image
# Panels 2-4 — For three example latents, the softmax attention weights over the 20% context pixels
HIGHLIGHT = [136, 50, 220]   # center-ish, top-left-ish, right-edge
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(img_show, interpolation='nearest')
lat_pix = coord_to_pixel(latent_pos)
axes[0].scatter(lat_pix[:, 1], lat_pix[:, 0], s=18, marker='o',
                facecolors='cyan', edgecolors='black', linewidths=0.7, alpha=0.85)
axes[0].set_title(f"All {NUM_LATENTS[0]} latent_pos anchors\n(16×16 grid + small jitter)")
axes[0].axis('off')

ctx_xy = sample['ctx_xy']
ctx_pix = coord_to_pixel(ctx_xy)
for i, lid in enumerate(HIGHLIGHT):
    lp  = latent_pos[lid]
    d2  = (ctx_xy - lp).pow(2).sum(-1)
    bias= -d2 / (2 * POS_BIAS_SIGMA**2)
    w   = F.softmax(bias, dim=0).numpy()

    axes[i+1].imshow(img_show * 0.25, interpolation='nearest')
    sc = axes[i+1].scatter(ctx_pix[:, 1], ctx_pix[:, 0], c=w, cmap='hot', s=22,
                           vmin=0, vmax=w.max(), edgecolors='gray', linewidths=0.3)
    lp_pix = coord_to_pixel(lp)
    axes[i+1].plot(lp_pix[1], lp_pix[0], '*', markersize=22, markerfacecolor='cyan',
                   markeredgecolor='black', markeredgewidth=1.5)
    axes[i+1].set_title(f"Latent #{lid} attention over\n20% context (σ={POS_BIAS_SIGMA}, ★=anchor)")
    axes[i+1].axis('off')
    plt.colorbar(sc, ax=axes[i+1], fraction=0.046)

plt.tight_layout(); plt.show()
print(f"\nEach latent's effective radius is ~σ = {POS_BIAS_SIGMA} in [-1,1]² coords.")
print(f"Latents whose anchors fall outside the context chunk get nearly uniform (low) attention")
print(f"— their content is mostly determined by the latent's own initialization.")


## Step 3 — The target features: deterministic Gaussian soft-pool

The target side has **zero learnable parameters beyond the encoder itself**. For each query
coordinate `q`, the target feature is a Gaussian-weighted average of the target encoder's latents:

$$
h_{\text{tgt}}[q] = \sum_l \frac{e^{-\|\,\text{latent\_pos}[l] - q\,\|^2 / 2\sigma_{\text{pool}}^2}}{\sum_{l'} e^{-\|\,\text{latent\_pos}[l'] - q\,\|^2 / 2\sigma_{\text{pool}}^2}} \cdot g_{\text{tgt}}[l]
$$

With `σ_pool = 0.15` this is *much tighter* than the encoder's `σ = 0.5` — only the 2-3 latents
nearest to the query coordinate contribute meaningfully. Effectively: "the target at position q is
roughly the content of the latent whose anchor is closest to q."

Because there's no learnable readout here (unlike v1-v4 which used an EMA'd `target_readout` module),
the target side cannot collapse independently of the encoder — it's a pure function of `g_tgt` and
`latent_pos`.


In [ ]:
# Visualize soft-pool weights for one target coord
def soft_pool_weights(latent_pos, query_coord, sigma):
    """Returns the softmax weights w[l] over latents for one query coord."""
    d2 = (latent_pos - query_coord).pow(2).sum(-1)
    return F.softmax(-d2 / (2 * sigma**2), dim=0).numpy()

# Show the soft-pool weights for three different target coords
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for col, q_idx in enumerate([0, 20, 50]):
    q_coord = sample['tgt_xy'][q_idx]
    w_pool  = soft_pool_weights(latent_pos, q_coord, POS_POOL_SIGMA)

    # Top row: spatial layout
    ax = axes[0, col]
    ax.imshow(img_show * 0.2, interpolation='nearest')
    ax.scatter(lat_pix[:, 1], lat_pix[:, 0],
               c=w_pool, cmap='hot',
               s=20 + 2000 * w_pool, alpha=0.85,
               vmin=0, vmax=w_pool.max(),
               edgecolors='gray', linewidths=0.3)
    q_pix = coord_to_pixel(q_coord)
    ax.plot(q_pix[1], q_pix[0], '*', markersize=24, markerfacecolor='red',
            markeredgecolor='white', markeredgewidth=2)
    ax.set_title(f"Target coord #{q_idx} at ({q_coord[0]:+.2f}, {q_coord[1]:+.2f})\nSoft-pool weights over latents (★ = query, σ_pool = {POS_POOL_SIGMA})")
    ax.axis('off')

    # Bottom row: histogram of top weights
    ax = axes[1, col]
    sorted_w = sorted(w_pool, reverse=True)
    ax.bar(range(20), sorted_w[:20], color='tab:red', alpha=0.7)
    ax.set_xlabel("Latents sorted by pool weight"); ax.set_ylabel("weight")
    top5 = sum(sorted_w[:5])
    ax.set_title(f"Top-20 latent weights\n(top-5 sum = {top5:.2f})")
    ax.grid(True, alpha=0.3); ax.set_xlim(-0.5, 19.5)

plt.tight_layout(); plt.show()
print("Verify: top-5 latent weights sum to >0.95 → soft-pool is essentially picking ~3 latents.")


## Step 4 — The predictor: learn to mimic the target from the 20 % view

The predictor is a small Perceiver-style module. It takes the **context encoder's latent set**
`g_ctx` (computed from the 20 % chunk only) plus a positional-encoded query `γ(target_coord)`, and
outputs `h_pred ∈ ℝ^(N_TGT, 512)`.

Architecturally:
- Project `g_ctx` to a bottleneck dim (`predictor_dim=256`).
- For each target coord, build a token = `mask_token + proj_query(γ(target_coord))`.
- Concatenate context tokens + target tokens; run 4 self-attn blocks; slice off the target slots; project back to 512.

The predictor **cannot see the target encoder's output** — only the context's latents. Its job is to
infer `h_tgt[q]` (which depends on `g_tgt`) from `g_ctx + γ(q)` alone.

Below we build a fresh, randomly-initialized model just for visualization (no training). The numbers
won't be meaningful yet — but the *shapes* and the *flow* will be clear.


In [ ]:
# ---- Minimal model definitions (matches JEPA_CIFAR10.ipynb) ----

class GaussianFourierFeatures(nn.Module):
    def __init__(self, in_features, mapping_size, scale=15.0):
        super().__init__()
        self.register_buffer('B', torch.randn((in_features, mapping_size)) * scale)
    def forward(self, coords):
        proj = coords @ self.B
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)


def get_sinusoidal_embeddings(n, d):
    position = torch.arange(n, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d, 2).float() * -(math.log(10000.0) / d))
    pe = torch.zeros(n, d)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class PreNorm(nn.Module):
    def __init__(self, dim, fn, context_dim=None):
        super().__init__()
        self.fn = fn; self.norm = nn.LayerNorm(dim)
        self.norm_context = nn.LayerNorm(context_dim) if context_dim is not None else None
    def forward(self, x, **kw):
        x = self.norm(x)
        if self.norm_context is not None: kw['context'] = self.norm_context(kw['context'])
        return self.fn(x, **kw)


class GEGLU(nn.Module):
    def forward(self, x):
        x, gates = x.chunk(2, dim=-1); return x * F.gelu(gates)


class FeedForward(nn.Module):
    def __init__(self, dim, mult=4):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim*mult*2), GEGLU(), nn.Linear(dim*mult, dim))
    def forward(self, x): return self.net(x)


class Attention(nn.Module):
    def __init__(self, query_dim, context_dim=None, heads=8, dim_head=64):
        super().__init__()
        inner = dim_head * heads
        context_dim = context_dim if context_dim is not None else query_dim
        self.scale, self.heads = dim_head ** -0.5, heads
        self.to_q   = nn.Linear(query_dim, inner, bias=False)
        self.to_kv  = nn.Linear(context_dim, inner * 2, bias=False)
        self.to_out = nn.Linear(inner, query_dim)
    def forward(self, x, context=None, mask=None, attn_bias=None):
        h = self.heads
        q = self.to_q(x)
        context = context if context is not None else x
        k, v = self.to_kv(context).chunk(2, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> (b h) n d', h=h), (q, k, v))
        sim = torch.einsum('b i d, b j d -> b i j', q, k) * self.scale
        if attn_bias is not None:
            B = x.shape[0]; H = sim.shape[0] // B
            ab = attn_bias.unsqueeze(1).expand(-1, H, -1, -1).reshape(B*H, *attn_bias.shape[1:])
            sim = sim + ab
        attn = sim.softmax(dim=-1)
        out = torch.einsum('b i j, b j d -> b i d', attn, v)
        return self.to_out(rearrange(out, '(b h) n d -> b n (h d)', h=h))


class CascadedBlock(nn.Module):
    def __init__(self, dim, n_latents, input_dim, cross_heads, cross_dim_head,
                 self_heads, self_dim_head, residual_dim=None, pos_bias_sigma=POS_BIAS_SIGMA):
        super().__init__()
        self.latents = nn.Parameter(get_sinusoidal_embeddings(n_latents, dim), requires_grad=True)
        self.cross_attn = PreNorm(dim, Attention(dim, input_dim, heads=cross_heads, dim_head=cross_dim_head), context_dim=input_dim)
        self.self_attn  = PreNorm(dim, Attention(dim, heads=self_heads, dim_head=self_dim_head))
        self.residual_proj = nn.Linear(residual_dim, dim) if residual_dim and residual_dim != dim else None
        self.ff = PreNorm(dim, FeedForward(dim))
        self.pos_bias_sigma = pos_bias_sigma
    def forward(self, context, residual=None, latent_pos=None, input_coords=None):
        b = context.size(0)
        x = repeat(self.latents, 'n d -> b n d', b=b)
        attn_bias = None
        if latent_pos is not None and input_coords is not None:
            d2 = (latent_pos.unsqueeze(0).unsqueeze(2) - input_coords.unsqueeze(1)).pow(2).sum(-1)
            attn_bias = -d2 / (2.0 * self.pos_bias_sigma**2)
        x = self.cross_attn(x, context=context, attn_bias=attn_bias) + x
        if residual is not None:
            r = self.residual_proj(residual) if self.residual_proj is not None else residual
            x = x + r
        x = self.self_attn(x) + x
        x = self.ff(x) + x
        return x


class JEPAEncoder(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, latent_dims=LATENT_DIMS, num_latents=NUM_LATENTS,
                 cross_heads=4, cross_dim_head=128, self_heads=8, self_dim_head=128,
                 num_trunk_layers=4, pos_bias_sigma=POS_BIAS_SIGMA):
        super().__init__()
        self.encoder_blocks = nn.ModuleList()
        prev = None
        for d, n in zip(latent_dims, num_latents):
            self.encoder_blocks.append(
                CascadedBlock(d, n, input_dim, cross_heads, cross_dim_head,
                              self_heads, self_dim_head, residual_dim=prev, pos_bias_sigma=pos_bias_sigma)
            )
            prev = d
        final = latent_dims[-1]
        self.trunk = nn.ModuleList([
            nn.ModuleList([PreNorm(final, Attention(final, heads=self_heads, dim_head=self_dim_head)),
                           PreNorm(final, FeedForward(final))])
            for _ in range(num_trunk_layers)
        ])
        N_lat = num_latents[0]
        gs = max(2, int(math.ceil(math.sqrt(N_lat))))
        ys = torch.linspace(-1.0, 1.0, gs); xs = torch.linspace(-1.0, 1.0, gs)
        grid_y = ys.unsqueeze(1).expand(gs, gs).contiguous()
        grid_x = xs.unsqueeze(0).expand(gs, gs).contiguous()
        grid = torch.stack([grid_y, grid_x], dim=-1).reshape(-1, 2)
        grid = grid[:N_lat] if grid.shape[0] >= N_lat else torch.cat([grid, torch.zeros(N_lat - grid.shape[0], 2)], dim=0)
        grid = grid + 0.01 * torch.randn_like(grid)
        self.latent_pos = nn.Parameter(grid, requires_grad=True)
    def forward(self, data, input_coords):
        residual = None
        for block in self.encoder_blocks:
            residual = block(data, residual=residual, latent_pos=self.latent_pos, input_coords=input_coords)
        for sa, ff in self.trunk:
            residual = sa(residual) + residual
            residual = ff(residual) + residual
        return residual, self.latent_pos.unsqueeze(0).expand(data.size(0), -1, -1)


class CoordReadout(nn.Module):
    def __init__(self, latent_dim=EMBED_DIM, predictor_dim=PRED_DIM,
                 query_in_dim=POS_DIM, num_layers=4, heads=8, dim_head=32):
        super().__init__()
        self.proj_g     = nn.Linear(latent_dim, predictor_dim)
        self.proj_query = nn.Linear(query_in_dim, predictor_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.blocks = nn.ModuleList([
            nn.ModuleList([PreNorm(predictor_dim, Attention(predictor_dim, heads=heads, dim_head=dim_head)),
                           PreNorm(predictor_dim, FeedForward(predictor_dim))])
            for _ in range(num_layers)
        ])
        self.norm     = nn.LayerNorm(predictor_dim)
        self.proj_out = nn.Linear(predictor_dim, latent_dim)
    def forward(self, g, query_coords_enc):
        N_lat = g.shape[1]
        ctx_tok = self.proj_g(g)
        tgt_tok = self.proj_query(query_coords_enc) + self.mask_token
        x = torch.cat([ctx_tok, tgt_tok], dim=1)
        for sa, ff in self.blocks:
            x = sa(x) + x; x = ff(x) + x
        x = self.norm(x)
        return self.proj_out(x[:, N_lat:])


def soft_pool_targets(g, latent_pos, query_coord, sigma):
    d2 = (query_coord.unsqueeze(2) - latent_pos.unsqueeze(0).unsqueeze(0)).pow(2).sum(-1)
    w  = F.softmax(-d2 / (2.0 * sigma**2), dim=-1)
    return torch.einsum('b q l, b l d -> b q d', w, g)


# Build a fresh model
torch.manual_seed(0)
fourier_encoder = GaussianFourierFeatures(2, FOURIER_DIM, scale=15.0)
context_encoder = JEPAEncoder()
predictor       = CoordReadout()
target_encoder  = copy.deepcopy(context_encoder)
for p in target_encoder.parameters(): p.requires_grad_(False)

n_enc  = sum(p.numel() for p in context_encoder.parameters()) / 1e6
n_pred = sum(p.numel() for p in predictor.parameters()) / 1e6
print(f"context_encoder: {n_enc:.2f}M params   predictor: {n_pred:.2f}M params (randomly initialized)")


In [ ]:
# Run one forward pass on the sample (CPU OK since just one sample)
def encode_input(pixels, coords):
    return torch.cat([pixels, fourier_encoder(coords)], dim=-1)

ctx_p = sample['pix_all'][sample['ctx_idx']].unsqueeze(0)       # (1, 205, 3)
ctx_c = sample['ctx_xy'].unsqueeze(0)                            # (1, 205, 2) — placeholder; use ctx_idx coords
ctx_c = coords_all[sample['ctx_idx']].unsqueeze(0)
pool_p = sample['pix_all'][sample['pool_idx']].unsqueeze(0)      # (1, 410, 3)
pool_c = coords_all[sample['pool_idx']].unsqueeze(0)
tgt_c  = sample['tgt_xy'].unsqueeze(0)                            # (1, 64, 2)

ctx_data = encode_input(ctx_p, ctx_c)
pool_data = encode_input(pool_p, pool_c)

with torch.no_grad():
    g_ctx, _              = context_encoder(ctx_data, ctx_c)        # (1, 256, 512)
    g_tgt, lat_pos_tgt    = target_encoder(pool_data, pool_c)        # (1, 256, 512), (1, 256, 2)
    h_tgt_raw             = soft_pool_targets(g_tgt, target_encoder.latent_pos, tgt_c, POS_POOL_SIGMA)  # (1, 64, 512)
    h_tgt_normed          = F.layer_norm(h_tgt_raw - h_tgt_raw.mean(dim=(0,1), keepdim=True), (EMBED_DIM,))
    tgt_pos_enc           = fourier_encoder(tgt_c)
    h_pred                = predictor(g_ctx, tgt_pos_enc)            # (1, 64, 512)

print(f"g_ctx:        {tuple(g_ctx.shape)}  (per-image content latents)")
print(f"g_tgt:        {tuple(g_tgt.shape)}  (target encoder, sees more)")
print(f"h_tgt_raw:    {tuple(h_tgt_raw.shape)} (soft-pool over g_tgt at target coords)")
print(f"h_tgt_normed: {tuple(h_tgt_normed.shape)} (centered + LayerNormed — what the predictor matches)")
print(f"h_pred:       {tuple(h_pred.shape)} (predictor output)")

# Per-coord loss
loss_per_q  = F.smooth_l1_loss(h_pred, h_tgt_normed, reduction='none').mean(dim=-1).squeeze(0).numpy()
cos_per_q   = F.cosine_similarity(h_pred, h_tgt_normed, dim=-1).squeeze(0).numpy()
print(f"\nMean smooth_l1 loss: {loss_per_q.mean():.4f}   Mean cos: {cos_per_q.mean():+.3f}")


In [ ]:
# Visualize per-target-coord predictions and loss
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel (top-left): per-coord smooth_l1 on the image
ax = axes[0, 0]
ax.imshow(img_show * 0.25, interpolation='nearest')
tgt_pix = coord_to_pixel(sample['tgt_xy'])
sc = ax.scatter(tgt_pix[:, 1], tgt_pix[:, 0], c=loss_per_q, cmap='Reds', s=80,
                edgecolors='white', linewidths=1)
plt.colorbar(sc, ax=ax)
ax.set_title("Per-target-coord smooth_l1 loss\n(red = predictor mismatch, fresh init = ~uniform)")
ax.axis('off')

# Panel (top-right): per-coord cosine similarity
ax = axes[0, 1]
ax.imshow(img_show * 0.25, interpolation='nearest')
sc = ax.scatter(tgt_pix[:, 1], tgt_pix[:, 0], c=cos_per_q, cmap='RdBu_r', s=80,
                vmin=-0.3, vmax=0.3, edgecolors='white', linewidths=1)
plt.colorbar(sc, ax=ax)
ax.set_title("Per-target-coord cos(h_pred, h_tgt)\n(blue = aligned, red = anti-aligned)")
ax.axis('off')

# Panel (bottom-left): heatmap of h_pred[:5] and h_tgt_normed[:5]
ax = axes[1, 0]
N_show = 12; D_show = 64
combined = torch.cat([h_pred[0, :N_show, :D_show], h_tgt_normed[0, :N_show, :D_show]], dim=0).numpy()
im = ax.imshow(combined, cmap='RdBu_r', vmin=-3, vmax=3, aspect='auto')
ax.axhline(y=N_show - 0.5, color='black', linewidth=2)
ax.set_yticks([N_show / 2 - 0.5, N_show * 1.5 - 0.5])
ax.set_yticklabels(['h_pred[:12]', 'h_tgt_normed[:12]'])
ax.set_xlabel(f"feature dim (showing first {D_show} of {EMBED_DIM})")
ax.set_title("Predictor vs target features\n(first 12 target coords × first 64 feature dims)")
plt.colorbar(im, ax=ax)

# Panel (bottom-right): cosine sim distribution
ax = axes[1, 1]
ax.hist(cos_per_q, bins=20, color='tab:blue', alpha=0.7, edgecolor='black')
ax.axvline(cos_per_q.mean(), color='red', linestyle='--', label=f'mean={cos_per_q.mean():+.3f}')
ax.axvline(0, color='gray', linestyle=':', label='random alignment')
ax.set_xlabel("cos(h_pred, h_tgt)"); ax.set_ylabel("count")
ax.set_title(f"Cosine distribution across {N_TGT} target coords\n(fresh model = near zero; trained = positive)")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()


## Step 5 — The full pipeline in one schematic

```
┌──── per-image, fixed by seed ─────┐    ┌── per-iteration random ─┐    ┌── per-iteration random ──┐
│       40% pool (410 px)           │    │  20% context chunk      │    │  N_TGT=64 target coords  │
│       seen by target encoder      │    │  (nearest to anchor)    │    │  (random subset of pool) │
└─────────────────┬─────────────────┘    └────────────┬────────────┘    └────────────┬─────────────┘
                  │                                   │                              │
                  ▼                                   ▼                              │
┌─────────────────────────────────┐  ┌────────────────────────────────┐               │
│ target_encoder (frozen, EMA)    │  │ context_encoder (TRAINED)      │               │
│ + Gaussian attn bias σ=0.5      │  │ + Gaussian attn bias σ=0.5     │               │
│   → g_tgt ∈ ℝ^(256, 512)        │  │   → g_ctx ∈ ℝ^(256, 512)       │               │
│   shares latent_pos with ctx    │  │   latent_pos: learnable        │               │
└────────────────┬────────────────┘  └────────────────┬───────────────┘               │
                 │                                    │                               │
                 ▼                                    │                               │
┌─────────────────────────────────┐                   │                               │
│ soft_pool_targets:              │                   │                               │
│ Gaussian σ=0.15 weighted avg    │ ←── target_coord ─┼───────────────────────────────┘
│ of g_tgt over latent_pos        │                   │
│ → h_tgt ∈ ℝ^(64, 512)           │                   │
│   (deterministic, no params)    │                   │
└────────────────┬────────────────┘                   │
                 │                                    │
       (center subtract + LN)                         │
                 │                                    ▼
                 │                ┌───────────────────────────────────────┐
                 │                │ predictor (TRAINED)                   │
                 │                │ - bottleneck proj_g: 512 → 256        │
                 │                │ - mask_token + proj_query(γ(coord))   │
                 │                │ - 4 self-attn blocks                  │
                 │                │ - proj_out: 256 → 512                 │
                 │                │ → h_pred ∈ ℝ^(64, 512)                │
                 │                └────────────────┬──────────────────────┘
                 │                                 │
                 ▼                                 ▼
                 ┌─────────────────────────────────┐
                 │  loss = smooth_l1(h_pred, sg(LN(h_tgt − center)))
                 └─────────────────────────────────┘
                              │
                       backward through:
                       - context_encoder
                       - predictor
                       (target_encoder updated via EMA m=0.999 → 1.0)
```

### Why "premature convergence" is the failure mode

After ~5-10 epochs the model finds a stable but uninformative regime where:

- `cos(h_pred, h_tgt)` saturates around 0.80 ± 0.13 (good-but-not-great alignment),
- `Lj` plateaus around 0.10 - 0.19 with no further improvement,
- `|g|` and `|h_t|` slowly inflate (encoder amplifies magnitudes without diversifying content),
- `σt(g)` keeps growing (encoder spreads latent tokens further apart),
- but linear-probe accuracy stays in the 22-29 % range.

The model has learned to *partially* predict the soft-pool target — enough to match it on average,
but the residual variance the predictor *would* need to capture to improve further is exactly the
fine-grained semantic content that would help the downstream classifier. Without an explicit anti-
collapse loss (VICReg-style variance/covariance), there's nothing forcing the encoder to use its
capacity on discriminative features. JEPA's bootstrap signal can plateau here even on much longer
schedules — i-JEPA's original paper trains 300 epochs for this reason, and even then it's known to
under-perform DINO on classification.
